# HR Attrition Analysis
**Author:** Vinay Kumar  
**Tools:** Python · Pandas · Matplotlib · Seaborn · SQL (SQLite)  
**Dataset:** 1,200 employee records across 6 departments

---

## Objective
Analyze employee attrition patterns to identify high-risk segments and key drivers of turnover.  
Deliver actionable recommendations for HR leadership to reduce attrition proactively.

---

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#F8FAFC'
plt.rcParams['font.family'] = 'DejaVu Sans'
NAVY = '#1F4E79'
ACCENT = '#2E75B6'
RED = '#C00000'
GREEN = '#375623'
LIGHT = '#BDD7EE'

df = pd.read_csv('HR_Attrition_Dataset.csv')
print(f'Dataset shape: {df.shape}')
print(f'Attrition rate: {df["Attrition"].value_counts(normalize=True)["Yes"]:.1%}')
df.head()

## 2. Dataset Overview

In [ ]:
print('=== DATASET INFO ===')
print(f'Total Employees : {len(df)}')
print(f'Total Features  : {len(df.columns)}')
print(f'Departments     : {df["Department"].nunique()} ({", ".join(df["Department"].unique())})')
print(f'Cities          : {df["City"].nunique()} ({", ".join(df["City"].unique())})')
print(f'Missing Values  : {df.isnull().sum().sum()}')
print()
print('=== ATTRITION BREAKDOWN ===')
print(df['Attrition'].value_counts())
print(df['Attrition'].value_counts(normalize=True).apply(lambda x: f'{x:.1%}'))

## 3. SQL Analysis with SQLite
All key business questions answered using SQL queries.

In [ ]:
# Load into SQLite
conn = sqlite3.connect(':memory:')
df.to_sql('HR_Attrition', conn, index=False)

# Q1: Overall attrition rate
q1 = pd.read_sql('''
    SELECT COUNT(*) AS TotalEmployees,
           SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END) AS AttritionCount,
           ROUND(SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END)*100.0/COUNT(*),2) AS AttritionRate_Pct
    FROM HR_Attrition
''', conn)
print('--- Q1: Overall Attrition Rate ---')
print(q1.to_string(index=False))

In [ ]:
# Q2: Attrition by Department
q2 = pd.read_sql('''
    SELECT Department, COUNT(*) AS Total,
           SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END) AS Attrited,
           ROUND(SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END)*100.0/COUNT(*),2) AS AttritionRate_Pct
    FROM HR_Attrition
    GROUP BY Department ORDER BY AttritionRate_Pct DESC
''', conn)
print('--- Q2: Attrition by Department ---')
print(q2.to_string(index=False))

In [ ]:
# Q3: Attrition by Overtime
q3 = pd.read_sql('''
    SELECT OverTime, COUNT(*) AS Total,
           SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END) AS Attrited,
           ROUND(SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END)*100.0/COUNT(*),2) AS AttritionRate_Pct
    FROM HR_Attrition GROUP BY OverTime
''', conn)
print('--- Q3: Attrition by Overtime ---')
print(q3.to_string(index=False))

In [ ]:
# Q4: Attrition by Tenure Band
q4 = pd.read_sql('''
    SELECT CASE WHEN YearsAtCompany<=2 THEN '0-2 Years'
                WHEN YearsAtCompany BETWEEN 3 AND 5 THEN '3-5 Years'
                WHEN YearsAtCompany BETWEEN 6 AND 10 THEN '6-10 Years'
                ELSE '10+ Years' END AS TenureBand,
           COUNT(*) AS Total,
           SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END) AS Attrited,
           ROUND(SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END)*100.0/COUNT(*),2) AS AttritionRate_Pct
    FROM HR_Attrition GROUP BY TenureBand ORDER BY AttritionRate_Pct DESC
''', conn)
print('--- Q4: Attrition by Tenure Band ---')
print(q4.to_string(index=False))

In [ ]:
# Q5: High-Risk Employee Identification (Risk Scoring Model)
q5 = pd.read_sql('''
    SELECT EmployeeID, Department, JobRole, MonthlyIncome,
           (CASE WHEN JobSatisfaction <= 2 THEN 1 ELSE 0 END +
            CASE WHEN WorkLifeBalance <= 2 THEN 1 ELSE 0 END +
            CASE WHEN OverTime = 'Yes' THEN 1 ELSE 0 END +
            CASE WHEN MonthlyIncome < 50000 THEN 1 ELSE 0 END +
            CASE WHEN YearsAtCompany <= 2 THEN 1 ELSE 0 END) AS RiskScore
    FROM HR_Attrition
    WHERE Attrition = 'No'
    HAVING RiskScore >= 3
    ORDER BY RiskScore DESC
    LIMIT 10
''', conn)
print(f'--- Q5: Top 10 High-Risk Active Employees (RiskScore >= 3) ---')
print(q5.to_string(index=False))

## 4. Visualizations

In [ ]:
# --- Chart 1: Overall KPI Summary ---
total = len(df)
attrited = df['Attrition'].value_counts()['Yes']
retained = total - attrited
rate = attrited / total * 100

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle('HR Attrition — KPI Summary', fontsize=16, fontweight='bold', color=NAVY, y=1.02)

for ax, value, label, color in zip(
    axes,
    [total, attrited, f'{rate:.1f}%'],
    ['Total Employees', 'Employees Attrited', 'Attrition Rate'],
    [ACCENT, RED, NAVY]
):
    ax.set_facecolor('#F0F6FF')
    ax.text(0.5, 0.55, str(value), ha='center', va='center', fontsize=36,
            fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.25, label, ha='center', va='center', fontsize=12,
            color=NAVY, transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(2)

plt.tight_layout()
plt.savefig('chart1_kpi_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Chart 2: Attrition by Department ---
dept = df.groupby('Department')['Attrition'].apply(lambda x: (x=='Yes').sum()*100/len(x)).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = [RED if v > 40 else ACCENT for v in dept.values]
bars = ax.barh(dept.index, dept.values, color=colors, edgecolor='white', height=0.6)
for bar, val in zip(bars, dept.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%',
            va='center', fontsize=11, fontweight='bold', color=NAVY)
ax.set_xlabel('Attrition Rate (%)', fontsize=11, color=NAVY)
ax.set_title('Attrition Rate by Department', fontsize=14, fontweight='bold', color=NAVY, pad=15)
ax.axvline(x=df['Attrition'].apply(lambda x: 1 if x=='Yes' else 0).mean()*100,
           color='gray', linestyle='--', linewidth=1.2, label='Overall Average')
ax.legend(fontsize=10)
ax.set_xlim(0, 55)
plt.tight_layout()
plt.savefig('chart2_attrition_by_dept.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Chart 3: Overtime vs Attrition ---
ot = df.groupby('OverTime')['Attrition'].apply(lambda x: (x=='Yes').sum()*100/len(x))

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(ot.index, ot.values, color=[GREEN, RED], width=0.4, edgecolor='white')
for bar, val in zip(bars, ot.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.8, f'{val:.1f}%',
            ha='center', fontsize=13, fontweight='bold', color=NAVY)
ax.set_ylabel('Attrition Rate (%)', fontsize=11, color=NAVY)
ax.set_title('Overtime vs Attrition Rate', fontsize=14, fontweight='bold', color=NAVY, pad=15)
ax.set_ylim(0, 60)
ax.set_xticklabels(['No Overtime', 'Works Overtime'], fontsize=11)
plt.tight_layout()
plt.savefig('chart3_overtime_attrition.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Chart 4: Attrition by Tenure Band ---
df['TenureBand'] = pd.cut(df['YearsAtCompany'], bins=[-1, 2, 5, 10, 100],
                           labels=['0-2 Years', '3-5 Years', '6-10 Years', '10+ Years'])
tenure = df.groupby('TenureBand', observed=True)['Attrition'].apply(
    lambda x: (x=='Yes').sum()*100/len(x))

fig, ax = plt.subplots(figsize=(8, 5))
colors = [RED if v > 50 else ACCENT if v > 38 else LIGHT for v in tenure.values]
bars = ax.bar(tenure.index, tenure.values, color=colors, width=0.5, edgecolor='white')
for bar, val in zip(bars, tenure.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.8, f'{val:.1f}%',
            ha='center', fontsize=12, fontweight='bold', color=NAVY)
ax.set_ylabel('Attrition Rate (%)', fontsize=11, color=NAVY)
ax.set_title('Attrition Rate by Tenure Band\n(Critical: 0-2 Year Employees)', fontsize=13, fontweight='bold', color=NAVY, pad=15)
ax.set_ylim(0, 70)
plt.tight_layout()
plt.savefig('chart4_attrition_by_tenure.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Chart 5: Job Satisfaction vs Attrition ---
sat = df.groupby('JobSatisfaction')['Attrition'].apply(lambda x: (x=='Yes').sum()*100/len(x))
labels = {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'}

fig, ax = plt.subplots(figsize=(8, 5))
colors = [RED, '#E06C2B', ACCENT, GREEN]
bars = ax.bar([labels[i] for i in sat.index], sat.values, color=colors, width=0.5, edgecolor='white')
for bar, val in zip(bars, sat.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.8, f'{val:.1f}%',
            ha='center', fontsize=12, fontweight='bold', color=NAVY)
ax.set_ylabel('Attrition Rate (%)', fontsize=11, color=NAVY)
ax.set_title('Attrition Rate by Job Satisfaction Level', fontsize=13, fontweight='bold', color=NAVY, pad=15)
ax.set_ylim(0, 65)
plt.tight_layout()
plt.savefig('chart5_job_satisfaction.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Chart 6: Income Band vs Attrition ---
df['IncomeBand'] = pd.cut(df['MonthlyIncome'], bins=[0, 50000, 100000, 150000, 999999],
                           labels=['Below 50K', '50K-1L', '1L-1.5L', 'Above 1.5L'])
income = df.groupby('IncomeBand', observed=True)['Attrition'].apply(
    lambda x: (x=='Yes').sum()*100/len(x))

fig, ax = plt.subplots(figsize=(8, 5))
colors = [RED, ACCENT, LIGHT, '#90C3D4']
bars = ax.bar(income.index, income.values, color=colors, width=0.5, edgecolor='white')
for bar, val in zip(bars, income.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.8, f'{val:.1f}%',
            ha='center', fontsize=12, fontweight='bold', color=NAVY)
ax.set_ylabel('Attrition Rate (%)', fontsize=11, color=NAVY)
ax.set_title('Attrition Rate by Monthly Income Band', fontsize=13, fontweight='bold', color=NAVY, pad=15)
ax.set_ylim(0, 60)
plt.tight_layout()
plt.savefig('chart6_income_band.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Chart 7: Heatmap — Satisfaction x Work-Life Balance ---
pivot = df.pivot_table(values='Attrition', index='JobSatisfaction',
                        columns='WorkLifeBalance',
                        aggfunc=lambda x: (x=='Yes').sum()*100/len(x))
sat_labels = {1:'Low', 2:'Medium', 3:'High', 4:'Very High'}
pivot.index = [sat_labels[i] for i in pivot.index]
pivot.columns = [sat_labels[i] for i in pivot.columns]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Attrition Rate (%)'})
ax.set_title('Attrition Rate Heatmap\nJob Satisfaction × Work-Life Balance',
             fontsize=13, fontweight='bold', color=NAVY, pad=15)
ax.set_xlabel('Work-Life Balance', fontsize=11, color=NAVY)
ax.set_ylabel('Job Satisfaction', fontsize=11, color=NAVY)
plt.tight_layout()
plt.savefig('chart7_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Risk Scoring Model
Flagging currently active employees most likely to leave.

In [ ]:
active = df[df['Attrition'] == 'No'].copy()
active['RiskScore'] = (
    (active['JobSatisfaction'] <= 2).astype(int) +
    (active['WorkLifeBalance'] <= 2).astype(int) +
    (active['OverTime'] == 'Yes').astype(int) +
    (active['MonthlyIncome'] < 50000).astype(int) +
    (active['YearsAtCompany'] <= 2).astype(int)
)

high_risk = active[active['RiskScore'] >= 3].sort_values('RiskScore', ascending=False)
print(f'Total high-risk active employees (RiskScore >= 3): {len(high_risk)}')
print()
print('Risk Score Distribution:')
print(high_risk['RiskScore'].value_counts().sort_index(ascending=False))
print()
print('High-Risk Employees by Department:')
print(high_risk['Department'].value_counts())

In [ ]:
# --- Chart 8: Risk Score Distribution ---
risk_dist = active['RiskScore'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
colors = [GREEN if i < 3 else RED for i in risk_dist.index]
bars = ax.bar(risk_dist.index, risk_dist.values, color=colors, width=0.6, edgecolor='white')
for bar, val in zip(bars, risk_dist.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 2, str(val),
            ha='center', fontsize=11, fontweight='bold', color=NAVY)
ax.set_xlabel('Risk Score (0 = Safe, 5 = Critical)', fontsize=11, color=NAVY)
ax.set_ylabel('Number of Employees', fontsize=11, color=NAVY)
ax.set_title('Employee Risk Score Distribution\n(Red = High Risk, Score ≥ 3)', fontsize=13, fontweight='bold', color=NAVY)
ax.set_xticks(risk_dist.index)
safe_patch = mpatches.Patch(color=GREEN, label='Low Risk (Score 0-2)')
risk_patch = mpatches.Patch(color=RED, label='High Risk (Score 3-5)')
ax.legend(handles=[safe_patch, risk_patch], fontsize=10)
plt.tight_layout()
plt.savefig('chart8_risk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Key Findings & Recommendations

In [ ]:
print('=' * 60)
print('KEY FINDINGS')
print('=' * 60)
print()
print('1. OVERALL ATTRITION')
print(f'   37.9% attrition rate across 1,200 employees (455 left)')
print()
print('2. DEPARTMENT RISK')
print('   Sales (42.1%) and Marketing (40.8%) are highest risk')
print('   HR (31.1%) is most stable')
print()
print('3. OVERTIME IS THE BIGGEST LEVER')
print('   Employees with overtime: 46.1% attrition')
print('   Employees without overtime: 33.4% attrition')
print('   Difference: +12.7 percentage points')
print()
print('4. EARLY TENURE IS CRITICAL')
print('   0-2 year employees: 54.2% attrition (highest group)')
print('   10+ year employees: 35.8% attrition')
print()
print('5. SATISFACTION DRIVES DECISIONS')
print('   Low satisfaction: 49.5% attrition')
print('   Very High satisfaction: 27.8% attrition')
print()
print('6. HIGH-RISK EMPLOYEES IDENTIFIED')
print(f'   {len(high_risk)} active employees flagged with RiskScore >= 3')
print()
print('=' * 60)
print('RECOMMENDATIONS')
print('=' * 60)
print()
print('R1. Launch early-tenure retention programme (0-2 year focus)')
print('R2. Review overtime policies in Sales and Operations')
print('R3. Quarterly pulse surveys on satisfaction and work-life balance')
print('R4. Salary benchmarking for sub-50K income band')
print('R5. Proactive HR conversations with 87 flagged high-risk employees')

---
**Built by Vinay Kumar**  
[LinkedIn](https://www.linkedin.com/in/vinaykumar2412/) · [GitHub](https://github.com/Vinayjindhad)